# Tutorial 01 — Analyze Xenium Data

**Source:** https://squidpy.readthedocs.io/en/stable/notebooks/tutorials/tutorial_xenium.html  
**Dataset:** Human Lung Cancer (10x Xenium)  
**Cells:** 161,000 | **Genes:** 480

---

## What this notebook covers
1. Load Xenium data via `spatialdata-io` and convert to Zarr
2. Quality control metrics
3. Standard scRNA-seq preprocessing → UMAP → Leiden clustering
4. Visualize clusters in spatial coordinates
5. Spatial statistics: Moran's I, co-occurrence, neighborhood enrichment, centrality scores
6. Interactive visualization with `napari-spatialdata`

## 0. Imports

In [ ]:
import spatialdata as sd
from spatialdata_io import xenium

import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import squidpy as sq

import warnings
warnings.filterwarnings('ignore')

sc.logging.print_header()
print(f"squidpy=={sq.__version__}")

## 1. Load Xenium Data

Download the [Xenium Human Lung Cancer dataset](https://www.10xgenomics.com/datasets/preview-data-ffpe-human-lung-cancer-with-xenium-multimodal-cell-segmentation-1-standard) from 10x Genomics and extract it into `../data/Xenium/`.

The `xenium()` reader from `spatialdata-io` parses the raw Xenium output into a `SpatialData` object containing:
- **Images:** morphology focus image (multiscale)
- **Labels:** cell and nucleus segmentation masks
- **Points:** individual transcript locations (40M+ transcripts)
- **Shapes:** cell/nucleus boundary polygons
- **Tables:** AnnData with count matrix (161,000 cells × 480 genes)

In [ ]:
xenium_path = "../data/Xenium"
zarr_path   = "../data/Xenium.zarr"

# Parse Xenium output into SpatialData
sdata = xenium(xenium_path)

# Write to Zarr (efficient on-disk format for large datasets)
sdata.write(zarr_path)

# Read directly from Zarr store going forward
sdata = sd.read_zarr(zarr_path)
print(sdata)

In [ ]:
# Extract the AnnData table for standard scRNA-seq-style analysis
adata = sdata.tables["table"]
print(adata)
print("\nCell metadata columns:")
print(adata.obs.columns.tolist())

## 2. Quality Control

In [ ]:
# Visualize QC metric distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(adata.obs['transcript_counts'], bins=100)
axes[0].set_xlabel('Transcript counts')
axes[0].set_ylabel('Cells')
axes[0].set_title('Transcript counts per cell')

axes[1].hist(adata.obs['cell_area'], bins=100)
axes[1].set_xlabel('Cell area (µm²)')
axes[1].set_title('Cell area distribution')

axes[2].scatter(adata.obs['cell_area'], adata.obs['transcript_counts'],
                alpha=0.1, s=1)
axes[2].set_xlabel('Cell area (µm²)')
axes[2].set_ylabel('Transcript counts')
axes[2].set_title('Area vs transcript counts')

plt.tight_layout()
plt.savefig('../results/figures/01_xenium/qc_metrics.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Filter low-quality cells
print(f"Cells before filtering: {adata.n_obs}")
sc.pp.filter_cells(adata, min_counts=10)
print(f"Cells after filtering:  {adata.n_obs}")

## 3. Preprocessing → UMAP → Clustering

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, flavor='seurat_v3', n_top_genes=3000)
sc.tl.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.tl.leiden(adata, key_added='leiden')

print(f"Leiden clusters: {adata.obs['leiden'].nunique()}")

## 4. Visualize Clusters — UMAP and Spatial

In [ ]:
sc.pl.umap(adata, color='leiden', save='_xenium_leiden.png')

In [ ]:
sq.pl.spatial_scatter(
    adata,
    color='leiden',
    shape=None,
    size=2,
    save='../results/figures/01_xenium/spatial_leiden.png'
)

## 5. Spatial Statistics

### 5a. Neighborhood Enrichment
Tests whether cell types are spatially co-localized more than expected by chance.

In [ ]:
sq.gr.spatial_neighbors(adata, coord_type='generic', delaunay=True)
sq.gr.nhood_enrichment(adata, cluster_key='leiden')
sq.pl.nhood_enrichment(
    adata,
    cluster_key='leiden',
    method='average',
    save='../results/figures/01_xenium/nhood_enrichment.png'
)

### 5b. Co-occurrence Score

In [ ]:
sq.gr.co_occurrence(adata, cluster_key='leiden')
sq.pl.co_occurrence(
    adata,
    cluster_key='leiden',
    clusters='0',
    save='../results/figures/01_xenium/co_occurrence.png'
)

### 5c. Spatial Autocorrelation — Moran's I

In [ ]:
sq.gr.spatial_autocorr(adata, mode='moran')

# Top spatially variable genes
top_genes = adata.uns['moranI'].head(10)
print("Top 10 spatially variable genes (Moran's I):")
print(top_genes)

### 5d. Centrality Scores

In [ ]:
sq.gr.centrality_scores(adata, cluster_key='leiden')
sq.pl.centrality_scores(
    adata,
    cluster_key='leiden',
    save='../results/figures/01_xenium/centrality_scores.png'
)

### 5e. Interaction Matrix

In [ ]:
sq.gr.interaction_matrix(adata, cluster_key='leiden')
sq.pl.interaction_matrix(
    adata,
    cluster_key='leiden',
    save='../results/figures/01_xenium/interaction_matrix.png'
)

## 6. Save Results

In [ ]:
adata.write('../results/xenium_analyzed.h5ad')
print('Saved: ../results/xenium_analyzed.h5ad')